<a href="https://colab.research.google.com/github/YBenjaminPCondori/AI-ML-Learning/blob/main/attention_and_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
__author__ = "Pranava Madhyastha, Michael Akintunde"
__version__ = "INM434/IN3045 City, University of London, Spring 2026"

# Attention and Transformers

Today we will try take a peek into concepts of attention and the modelling framework called Transformers.

### TODO: Please print out the values and verify if you are able to understand each step below. Use lecture slides as the reference.

In [ ]:
import torch
import torch.nn.functional as F

# The sample sentence

We will begin with the following sample sentence: `the cat is on a new mat`

In [ ]:
# Define the input sentence
input_sentence = 'the cat is on a new mat'

# Creation of the dictionary

The next step is to create a dictionary that maps each word in the input sentence to a unique index. This is done by splitting the sentence into a list of words.

The list of words is then sorted, and each word is assigned an index using the `enumerate()` function. The result is a dictionary that maps each word to a unique index.

In [ ]:
# Create a dictionary that maps each word to a unique index
word_to_index = {word: i for i, word in enumerate(sorted(input_sentence.split()))}

# Tensor of indices

The input sentence is then converted to a tensor of indices using the word_to_index dictionary. The tensor is created by iterating over each word in the input sentence (after removing the comma) and looking up its index in the dictionary. The resulting tensor is a one-dimensional tensor of integer indices.

In [ ]:
# Convert the input sentence to a tensor of indices using the word_to_index dictionary
input_tensor = torch.tensor([word_to_index[word] for word in input_sentence.replace(',', '').split()])

# The embedding layer

The next step is to define an embedding layer. The embedding layer is a PyTorch module that learns a lookup table of word embeddings for a vocabulary of size n and embedding dimensionality d. In this case, the embedding layer is initialized with a vocabulary size equal to the number of words in the input sentence and an embedding dimensionality of 16. The input tensor is then embedded using the embedding layer to produce an embedded sentence. The `.detach()` method is used to detach the embedded sentence from its computation graph, preventing it from being backpropagated through.

In [ ]:
# Define the embedding layer
embedding_layer = torch.nn.Embedding(len(word_to_index), 16)

# Embed the input tensor to get the embedded sentence
embedded_sentence = embedding_layer(input_tensor).detach()

The size of the embedding is then defined by taking the shape of the embedded sentence and extracting its second dimension.

In [ ]:
# Define the size of the embedding
embedding_size = embedded_sentence.shape[1]

# Query, key and value

The sizes of the query, key, and value vectors are defined. Recall that these vectors are used in the attention mechanism to compute the attention weights and context vectors. In this case, the query and key vectors have a size of 24, while the value vector has a size of 28.

In [ ]:
# Define the sizes of the query, key, and value vectors
query_size, key_size, value_size = 24, 24, 28

# Initialisation and projection

Query, key, and value weight matrices are then defined using PyTorch's random number generator. The weight matrices are used to compute the query, key, and value vectors for each word in the input sentence. Refer to the lecture slides!

In [ ]:
# Define the query, key, and value weight matrices
query_weights = torch.rand(query_size, embedding_size)
key_weights = torch.rand(key_size, embedding_size)
value_weights = torch.rand(value_size, embedding_size)

# Example computation

The query, key, and value vectors for the second word in the input sentence are then computed by multiplying the corresponding weight matrices by the embedding vector for the second word.

In [ ]:
# Get the query, key, and value vectors for the second word in the input sentence
x_2 = embedded_sentence[1]
query_2 = query_weights.matmul(x_2)
key_2 = key_weights.matmul(x_2)
value_2 = value_weights.matmul(x_2)

# Parallelisation

The keys and values for all words in the input sentence are then computed by multiplying the corresponding weight matrices by the transpose of the embedded sentence. The transpose operation is used to swap the rows and columns of the embedded sentence, making it possible to compute the dot product between the weight matrices and the embedding vectors. The transpose operation is then applied again to swap the rows and columns back to their original order.

In [ ]:
# Compute the keys and values for all words in the input sentence
keys = key_weights.matmul(embedded_sentence.transpose(0, 1)).transpose(0, 1)
values = value_weights.matmul(embedded_sentence.transpose(0, 1)).transpose(0, 1)

# Computing the attention weights

The attention weights for the second word in the input sentence are then computed using the softmax function applied to the dot product of the query vector for the second word and the transpose of the key vectors for all words in the input sentence. The softmax function normalizes the dot product, resulting in a set of attention weights that sum to one. The size of the key vectors is used to scale the dot product.

In [ ]:
# Compute the keys and values for all words in the input sentence
keys = key_weights.matmul(embedded_sentence.transpose(0, 1)).transpose(0, 1)
values = value_weights.matmul(embedded_sentence.transpose(0, 1)).transpose(0, 1)

# Computing `h`, the context vector
The context vector for the second word in the input sentence is then computed by multiplying the attention weights by the value vectors for all words in the input sentence. The result is a weighted sum of the value vectors, where the weights are given by the attention weights.

In [ ]:
# Compute the attention weights for the second word in the input sentence
omega_2 = query_2.matmul(keys.T)
attention_weights_2 = F.softmax(omega_2 / key_size **0.5, dim=0)


# Compute the context vector for the second word in the input sentence
context_vector_2 = attention_weights_2.matmul(values)

# Multi-head attention
The code then defines the number of attention heads to use, which is set to 3. It also defines new query, key, and value weight matrices for the multi-head attention mechanism. The size of each weight matrix is expanded to include a third dimension for the number of attention heads.

Refer to the lecture notes to understand the concept of Multi-head attention.

In [ ]:
# Define the number of attention heads
num_heads = 3

# Define the query, key, and value weight matrices for the multi-head attention
multihead_query_weights = torch.rand(num_heads, query_size, embedding_size)
multihead_key_weights = torch.rand(num_heads, key_size, embedding_size)
multihead_value_weights = torch.rand(num_heads, value_size, embedding_size)

The query vector for the second word in the input sentence is obtained by multiplying the embedded vector of the second word with the multi-head query weight matrix. This is done separately for each attention head.

In [ ]:
# Get the query vector for the second word in the input sentence for each attention head
multihead_query_2 = multihead_query_weights.matmul(x_2)

# Repeat
The embedded sentence is repeated for each attention head using the repeat function. This allows each head to have its own set of keys and values for computing attention.

In [ ]:
# Repeat the embedded sentence for each attention head
repeated_embedded_sentence = embedded_sentence.unsqueeze(0).repeat(num_heads, 1, 1)

The keys and values for all words in the input sentence are computed for each attention head by multiplying the repeated embedded sentence with the multi-head key and value weight matrices. The transpose function is used to reshape the tensor dimensions so that the matrix multiplication can be computed.

In [ ]:
# Compute the keys and values for all words in the input sentence for each attention head
multihead_keys = multihead_key_weights.matmul(repeated_embedded_sentence.transpose(1, 2)).transpose(1, 2)
multihead_values = multihead_value_weights.matmul(repeated_embedded_sentence.transpose(1, 2)).transpose(1, 2)

# Compute the attention weights and context vectors for each attention head
attention_weights = F.softmax(multihead_query_2.matmul(multihead_keys.transpose(1, 2)) / key_size**0.5, dim=2)
context_vectors = attention_weights.matmul(multihead_values)

### TODO: Please follow *the annotated transformer*: http://nlp.seas.harvard.edu/annotated-transformer/ and consider re-writing and running it here on google colab.